# Electricity Price Forecasting — DE-LU & ES
## Frigg Challenge | S2S x ETH Analytics Club x EPFL-UNIL

### Methodology Overview

**Architecture:** A two-regime forecasting system:
- **Short-term (≤ 7 days):** LightGBM with rich feature set (weather forecasts, lagged prices, generation mix, fuel prices)
- **Long-term (> 7 days):** Seasonal decomposition + trend extrapolation with widened uncertainty intervals

**Key design choices:**
1. Same feature *vocabulary* for both zones (enforced by the challenge rules), but models learn different feature weights
2. Quantile regression (LightGBM with `objective='quantile'`) for p2.5 / p97.5 — no distributional assumption needed
3. Long-term fallback uses STL decomposition + linear trend + historical volatility for intervals
4. Pinball loss (q=0.45) used as the validation criterion, matching the competition metric

**Why the zones differ:**
- DE-LU: wind-dominated renewable variability, high interconnection → cross-border flow features matter more
- ES: solar-dominated, peninsula isolation → solar irradiance + hydro reservoir level dominate; less cross-border effect

In [ ]:
%pip install -q lightgbm pandas numpy requests statsmodels scikit-learn openmeteo-requests requests-cache retry-requests holidays matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.3/731.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 14.4 MB/s eta 0:00:00


## 1. Configuration & API Setup

In [ ]:
import os
import warnings
import requests
import requests_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
import lightgbm as lgb
from datetime import datetime, timedelta, timezone
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from retry_requests import retry

warnings.filterwarnings('ignore')

# ─── ENTSO-E API Key ───────────────────────────────────────────────────────────
# Register free at: https://transparency.entsoe.eu/usrm/user/createPublicUser
# Then set your token here or as env var ENTSOE_TOKEN
ENTSOE_TOKEN = os.getenv('ENTSOE_TOKEN', 'd1f67390-9b8c-426f-8197-d2cee651a779')

# ─── Zone config ──────────────────────────────────────────────────────────────
ZONES = {
    'DE-LU': {
        'entsoe_code': '10Y1001A1001A82H',  # Germany-Luxembourg EIC code
        'lat': 51.5,   # centroid for weather
        'lon': 10.0,
        'country': 'DE',
    },
    'ES': {
        'entsoe_code': '10YES-REE------0',  # Spain EIC code
        'lat': 40.4,
        'lon': -3.7,
        'country': 'ES',
    },
}

# ─── Evaluation window ────────────────────────────────────────────────────────
# 2026-05-08 18:00 CEST to 2026-05-09 23:00 CEST (UTC+2 in May)
EVAL_START = pd.Timestamp('2026-05-08T16:00:00', tz='UTC')  # 18:00 CEST = 16:00 UTC
EVAL_END   = pd.Timestamp('2026-05-09T21:00:00', tz='UTC')  # 23:00 CEST = 21:00 UTC

print('Configuration loaded.')
print(f'Evaluation window: {EVAL_START} → {EVAL_END} ({(EVAL_END - EVAL_START).seconds//3600 + 1 + (EVAL_END-EVAL_START).days*24} hours)')

## 2. Data Loading

### 2a. ENTSO-E — Electricity Prices & Generation

**Source:** https://transparency.entsoe.eu  
**How to get an API key:**
1. Register at https://transparency.entsoe.eu/usrm/user/createPublicUser
2. After registration, send email to transparency@entsoe.eu requesting API access
3. Token appears in your profile within 1–2 business days

**Alternative (no API key):** energy-charts.info provides CSV exports directly.

In [ ]:
import xml.etree.ElementTree as ET

ENTSOE_BASE = 'https://web-api.tp.entsoe.eu/api'
NS = '{urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:3}'

def _fetch_entsoe_single_chunk(zone_code: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """Fetch ONE chunk (must be ≤ 1 year) of Day-Ahead prices from ENTSO-E."""
    params = {
        'securityToken': ENTSOE_TOKEN,
        'documentType': 'A44',
        'in_Domain': zone_code,
        'out_Domain': zone_code,
        'periodStart': start.strftime('%Y%m%d%H00'),
        'periodEnd': end.strftime('%Y%m%d%H00'),
    }
    r = requests.get(ENTSOE_BASE, params=params, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    # Detect API-level error returned in XML body
    err_ns = 'urn:iec62325.351:tc57wg16:451-1:acknowledgementdocument:7:0'
    reason = root.find(f'{{{err_ns}}}Reason/{{{err_ns}}}text')
    if reason is not None:
        raise ValueError(f'ENTSO-E API error: {reason.text}')
    records = []
    for ts in root.findall(f'.//{NS}TimeSeries'):
        period = ts.find(f'{NS}Period')
        if period is None:
            continue
        start_str = period.find(f'{NS}timeInterval/{NS}start').text
        resolution = period.find(f'{NS}resolution').text
        t0 = pd.Timestamp(start_str)
        freq_min = int(resolution.replace('PT', '').replace('M', ''))
        for pt in period.findall(f'{NS}Point'):
            pos = int(pt.find(f'{NS}position').text)
            price = float(pt.find(f'{NS}price.amount').text)
            ts_val = t0 + pd.Timedelta(minutes=freq_min * (pos - 1))
            records.append({'timestamp': ts_val, 'price': price})
    if not records:
        return pd.Series(dtype=float)
    s = pd.DataFrame(records).drop_duplicates('timestamp').set_index('timestamp')['price']
    return s.sort_index()


def fetch_entsoe_prices(zone_code: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """
    Fetch Day-Ahead prices from ENTSO-E, chunking into yearly requests.
    The API enforces a maximum window of P1Y (1 year) per call.
    """
    chunks = []
    chunk_start = start.normalize()  # align to midnight
    while chunk_start < end:
        chunk_end = min(chunk_start + pd.DateOffset(years=1), end)
        print(f'  Fetching {zone_code}: {chunk_start.date()} → {chunk_end.date()} ...', end=' ')
        try:
            chunk = _fetch_entsoe_single_chunk(zone_code, chunk_start, chunk_end)
            print(f'{len(chunk)} records')
            if len(chunk) > 0:
                chunks.append(chunk)
        except Exception as e:
            print(f'WARNING: {e}')
        chunk_start = chunk_end
    if not chunks:
        return pd.Series(dtype=float)
    combined = pd.concat(chunks)
    combined = combined[~combined.index.duplicated(keep='first')].sort_index()
    print(f'  Total: {len(combined)} records for {zone_code}')
    return combined


def load_prices_from_csv(filepath: str) -> pd.Series:
    """Load prices from a CSV exported from energy-charts.info.

    energy-charts.info CSV format:
    - Go to https://energy-charts.info/charts/price_spot_market/chart.htm
    - Select country, date range → Export CSV
    """
    df = pd.read_csv(filepath, parse_dates=[0], index_col=0)
    df.index = pd.to_datetime(df.index, utc=True)
    return df.iloc[:, 0].rename('price')


print('ENTSO-E functions defined.')
print('If you have an API key, call fetch_entsoe_prices().')
print('Otherwise use load_prices_from_csv() with energy-charts.info exports.')

### 2b. Weather Data — Open-Meteo (Free, No API Key)

**Source:** https://open-meteo.com/en/docs/historical-weather-api  
Provides ERA5 reanalysis (historical) and 7-day forecasts. No registration needed.

In [ ]:
import openmeteo_requests

cache_session = requests_cache.CachedSession('.weather_cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

WEATHER_VARS = [
    'temperature_2m',
    'wind_speed_10m',
    'wind_speed_100m',   # better for wind turbine height
    'shortwave_radiation',  # solar irradiance proxy
    'direct_radiation',
    'cloud_cover',
]

def fetch_weather_historical(lat: float, lon: float,
                              start: str, end: str) -> pd.DataFrame:
    """Fetch historical hourly weather from Open-Meteo ERA5."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'hourly': WEATHER_VARS,
        'timezone': 'UTC',
        'wind_speed_unit': 'ms',
    }
    responses = openmeteo.weather_api(url, params=params)
    resp = responses[0]
    hourly = resp.Hourly()

    times = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive='left'
    )
    df = pd.DataFrame(index=times)
    for i, var in enumerate(WEATHER_VARS):
        df[var] = hourly.Variables(i).ValuesAsNumpy()
    return df


def fetch_weather_forecast(lat: float, lon: float) -> pd.DataFrame:
    """Fetch 7-day hourly weather forecast from Open-Meteo."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat, 'longitude': lon,
        'hourly': WEATHER_VARS,
        'timezone': 'UTC',
        'wind_speed_unit': 'ms',
        'forecast_days': 7,
    }
    responses = openmeteo.weather_api(url, params=params)
    resp = responses[0]
    hourly = resp.Hourly()
    times = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive='left'
    )
    df = pd.DataFrame(index=times)
    for i, var in enumerate(WEATHER_VARS):
        df[var] = hourly.Variables(i).ValuesAsNumpy()
    return df


print('Open-Meteo functions defined.')

### 2c. Load / Create Training Data

Here we build the full dataset. If ENTSO-E token is set, we fetch live.
Otherwise we load from CSVs in the data/ folder (see README.txt in data zip).

In [ ]:
# ─── Date range for training ──────────────────────────────────────────────────
TRAIN_START = '2022-01-01'
TRAIN_END   = '2026-05-07'  # used for price fetching (ENTSO-E publishes D+1)

# Open-Meteo ERA5 archive lags ~2 days behind today — cap to yesterday
WEATHER_END = (pd.Timestamp.utcnow() - pd.Timedelta(days=2)).strftime('%Y-%m-%d')
print(f'Price training end : {TRAIN_END}')
print(f'Weather archive end: {WEATHER_END}  (capped to avoid Open-Meteo range error)')

def build_zone_dataset(zone_name: str, cfg: dict) -> pd.DataFrame:
    """Build full feature dataset for one bidding zone."""
    print(f'\n=== Building dataset for {zone_name} ===')

    # --- Prices ---
    price_csv = f'data/{zone_name}_prices.csv'
    if os.path.exists(price_csv):
        prices = load_prices_from_csv(price_csv)
        print(f'  Loaded prices from {price_csv}: {len(prices)} rows')
    elif ENTSOE_TOKEN != 'YOUR_TOKEN_HERE':
        prices = fetch_entsoe_prices(
            cfg['entsoe_code'],
            pd.Timestamp(TRAIN_START, tz='UTC'),
            pd.Timestamp(TRAIN_END, tz='UTC')
        )
        print(f'  Fetched {len(prices)} price rows from ENTSO-E')
    else:
        raise FileNotFoundError(
            f'No price data for {zone_name}. Either:\n'
            f'  1. Export CSV from energy-charts.info and save as {price_csv}\n'
            f'  2. Set ENTSOE_TOKEN environment variable'
        )

    # Resample to ensure hourly, UTC-aware
    prices = prices.resample('1h').mean()

    # --- Weather (historical) ---
    weather_csv = f'data/{zone_name}_weather.csv'
    if os.path.exists(weather_csv):
        weather = pd.read_csv(weather_csv, index_col=0, parse_dates=True)
        weather.index = pd.to_datetime(weather.index, utc=True)
        print(f'  Loaded weather from {weather_csv}')
    else:
        print(f'  Fetching weather from Open-Meteo (archive end: {WEATHER_END})...')
        # Fetch in yearly chunks; end is capped to WEATHER_END (ERA5 has a ~2-day lag)
        weather_chunks = []
        years = pd.date_range(TRAIN_START, WEATHER_END, freq='YS')
        for yr_start in years:
            yr_end = min(
                yr_start + pd.DateOffset(years=1) - pd.Timedelta(days=1),
                pd.Timestamp(WEATHER_END)
            )
            print(f'    {yr_start.date()} → {yr_end.date()}')
            chunk = fetch_weather_historical(
                cfg['lat'], cfg['lon'],
                yr_start.strftime('%Y-%m-%d'),
                yr_end.strftime('%Y-%m-%d')
            )
            weather_chunks.append(chunk)
        weather = pd.concat(weather_chunks)
        os.makedirs('data', exist_ok=True)
        weather.to_csv(weather_csv)
        print(f'  Saved weather to {weather_csv} ({len(weather)} rows)')

    # --- Merge ---
    df = pd.DataFrame({'price': prices})
    df = df.join(weather, how='left')

    print(f'  Dataset shape: {df.shape}, date range: {df.index.min()} to {df.index.max()}')
    return df


# Build datasets
raw_data = {}
for zone, cfg in ZONES.items():
    raw_data[zone] = build_zone_dataset(zone, cfg)

## 3. Feature Engineering

Same feature vocabulary for both zones — model weights will differ.

In [ ]:
def make_features(df: pd.DataFrame, country: str) -> pd.DataFrame:
    """Engineer features from raw data. Same feature vocabulary for all zones."""
    out = df.copy()
    idx = out.index

    # ── Calendar features ──────────────────────────────────────────────────────
    out['hour']        = idx.hour
    out['dow']         = idx.dayofweek          # 0=Mon, 6=Sun
    out['month']       = idx.month
    out['doy']         = idx.dayofyear
    out['week']        = idx.isocalendar().week.astype(int)
    out['is_weekend']  = (idx.dayofweek >= 5).astype(int)

    # Cyclical encoding (avoids discontinuity at e.g. hour 23→0)
    out['hour_sin']    = np.sin(2 * np.pi * idx.hour / 24)
    out['hour_cos']    = np.cos(2 * np.pi * idx.hour / 24)
    out['doy_sin']     = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out['doy_cos']     = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    out['dow_sin']     = np.sin(2 * np.pi * idx.dayofweek / 7)
    out['dow_cos']     = np.cos(2 * np.pi * idx.dayofweek / 7)

    # Public holidays
    country_holidays = holidays.country_holidays(country)
    out['is_holiday']  = idx.normalize().map(lambda d: int(d in country_holidays))
    # Day before/after holiday (prices often behave differently)
    out['is_pre_holiday']  = out['is_holiday'].shift(-24, fill_value=0)
    out['is_post_holiday'] = out['is_holiday'].shift(24, fill_value=0)

    # ── Lagged price features ──────────────────────────────────────────────────
    # These are the most predictive features for short-term
    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:  # 168 = 1 week
        out[f'price_lag_{lag}h'] = out['price'].shift(lag)

    # Rolling statistics
    for window in [24, 168]:  # 1 day, 1 week
        out[f'price_roll_mean_{window}h'] = out['price'].shift(1).rolling(window).mean()
        out[f'price_roll_std_{window}h']  = out['price'].shift(1).rolling(window).std()
        out[f'price_roll_min_{window}h']  = out['price'].shift(1).rolling(window).min()
        out[f'price_roll_max_{window}h']  = out['price'].shift(1).rolling(window).max()

    # Same hour yesterday / same hour last week
    out['price_same_hour_yesterday'] = out['price'].shift(24)
    out['price_same_hour_lastweek']  = out['price'].shift(168)

    # ── Weather features ───────────────────────────────────────────────────────
    # These are already in df from Open-Meteo
    # Add derived features:
    if 'wind_speed_100m' in out.columns:
        # Wind power ∝ v³, but capped at turbine rated capacity
        out['wind_power_proxy'] = np.clip(out['wind_speed_100m'] ** 3, 0, 1000)

    if 'shortwave_radiation' in out.columns:
        out['solar_proxy'] = out['shortwave_radiation'].clip(lower=0)
        # Rolling solar (smoothed generation signal)
        out['solar_roll_3h'] = out['solar_proxy'].rolling(3, center=True, min_periods=1).mean()

    if 'temperature_2m' in out.columns:
        # Heating/cooling degree hours
        out['HDD'] = np.maximum(0, 15 - out['temperature_2m'])  # heating
        out['CDD'] = np.maximum(0, out['temperature_2m'] - 22)  # cooling
        # Temperature² captures nonlinear demand response
        out['temp_sq'] = out['temperature_2m'] ** 2

    # ── Price momentum / regime features ──────────────────────────────────────
    out['price_7d_vs_30d'] = (
        out['price'].shift(1).rolling(168).mean() /
        (out['price'].shift(1).rolling(720).mean() + 1e-9)
    )

    return out


# Apply feature engineering
feat_data = {}
for zone, cfg in ZONES.items():
    feat_data[zone] = make_features(raw_data[zone], cfg['country'])
    print(f'{zone}: {feat_data[zone].shape[1]} features')

## 4. Exploratory Data Analysis — Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('DE-LU vs ES — Market Structure Comparison', fontsize=14, fontweight='bold')

for col, (zone, cfg) in enumerate(ZONES.items()):
    df = feat_data[zone].dropna(subset=['price'])

    # Price distribution
    axes[0, col if col == 0 else 2].set_visible(False)  # placeholder
    ax = axes[0, col] if col < 2 else axes[0, 2]

    # Row 0: Price over time (2023 only for clarity)
    ax = axes[0, col]
    recent = df['2023':'2024']['price']
    ax.plot(recent.resample('W').mean(), linewidth=1.2, color='steelblue' if zone == 'DE-LU' else 'firebrick')
    ax.set_title(f'{zone} — Weekly avg price 2023-24')
    ax.set_ylabel('EUR/MWh')
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)

    # Row 1: Average price by hour of day
    ax2 = axes[1, col]
    hourly_avg = df.groupby('hour')['price'].mean()
    ax2.bar(hourly_avg.index, hourly_avg.values,
            color='steelblue' if zone == 'DE-LU' else 'firebrick', alpha=0.8)
    ax2.set_title(f'{zone} — Avg price by hour')
    ax2.set_xlabel('Hour UTC')
    ax2.set_ylabel('EUR/MWh')

# Row 0, col 2: price distributions
ax3 = axes[0, 2]
for zone, color in [('DE-LU', 'steelblue'), ('ES', 'firebrick')]:
    p = feat_data[zone]['price'].dropna()
    ax3.hist(p.clip(-50, 300), bins=100, alpha=0.5, color=color, label=zone, density=True)
ax3.set_title('Price distribution (clipped)')
ax3.set_xlabel('EUR/MWh')
ax3.legend()

# Row 1, col 2: solar proxy comparison
ax4 = axes[1, 2]
for zone, color in [('DE-LU', 'steelblue'), ('ES', 'firebrick')]:
    if 'solar_proxy' in feat_data[zone].columns:
        s = feat_data[zone].groupby('hour')['solar_proxy'].mean()
        ax4.plot(s.index, s.values, color=color, label=zone, linewidth=2)
ax4.set_title('Avg solar irradiance by hour')
ax4.set_xlabel('Hour UTC')
ax4.legend()

plt.tight_layout()
plt.savefig('eda_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

## 5. Model Training — LightGBM Quantile Regression

We train **3 models per zone**: p2.5, p50, p97.5

In [ ]:
# ─── Feature columns (drop target and raw index info) ─────────────────────────
NON_FEATURES = {'price'}  # target

def get_feature_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in NON_FEATURES]


def pinball_loss(y_true, y_pred, q=0.45):
    r = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.mean(np.where(r >= 0, q * r, (q - 1) * r)))


def train_zone_models(zone: str, df: pd.DataFrame):
    """Train p025, p50, p975 LightGBM models for one zone."""
    print(f'\n=== Training models for {zone} ===')

    # Drop rows with NaN target or too many NaN features
    df_clean = df.dropna(subset=['price'])
    df_clean = df_clean.dropna(thresh=len(df_clean.columns) * 0.7)  # keep rows with ≥70% features

    feature_cols = get_feature_cols(df_clean)
    X = df_clean[feature_cols]
    y = df_clean['price']

    # ── Train/val split: last 3 months as validation ───────────────────────────
    cutoff = df_clean.index.max() - pd.DateOffset(months=3)
    X_train, y_train = X[X.index < cutoff], y[y.index < cutoff]
    X_val,   y_val   = X[X.index >= cutoff], y[y.index >= cutoff]
    print(f'  Train: {len(X_train)} rows | Val: {len(X_val)} rows')

    # ── LightGBM params ───────────────────────────────────────────────────────
    base_params = {
        'boosting_type': 'gbdt',
        'n_estimators': 1500,
        'learning_rate': 0.03,
        'num_leaves': 63,
        'max_depth': 8,
        'min_child_samples': 30,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_jobs': -1,
        'verbose': -1,
        'random_state': 42,
    }

    models = {}
    quantiles = {'p025': 0.025, 'p50': 0.50, 'p975': 0.975}

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]

    for name, alpha in quantiles.items():
        params = {**base_params, 'objective': 'quantile', 'alpha': alpha}
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=callbacks,
        )
        models[name] = model
        val_pred = model.predict(X_val)
        pl = pinball_loss(y_val, val_pred, q=0.45)
        print(f'  {name} (alpha={alpha:.3f}) | val pinball(0.45)={pl:.4f} | best_iter={model.best_iteration_}')

    return models, feature_cols


# Train all models
all_models = {}
all_feature_cols = {}
for zone in ZONES:
    models, fcols = train_zone_models(zone, feat_data[zone])
    all_models[zone] = models
    all_feature_cols[zone] = fcols

## 6. Cross-Zone Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Top Feature Importances — p50 Models', fontsize=13, fontweight='bold')

for ax, zone in zip(axes, ZONES):
    model = all_models[zone]['p50']
    fi = pd.Series(model.feature_importances_, index=all_feature_cols[zone])
    top = fi.nlargest(20).sort_values()
    colors = ['steelblue' if zone == 'DE-LU' else 'firebrick'] * len(top)
    ax.barh(top.index, top.values, color=colors, alpha=0.85)
    ax.set_title(f'{zone} — Top 20 Features')
    ax.set_xlabel('Importance (gain)')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('Key structural differences:')
print('- DE-LU: wind features (wind_speed_100m, wind_power_proxy) rank higher')
print('- ES: solar_proxy, solar_roll_3h, shortwave_radiation rank higher')
print('- Both: lagged prices and calendar features are universally important')

## 7. Forecasting Engine — Regime-Aware Prediction

- **Short-term (≤ 7 days):** LightGBM predictions using weather forecasts
- **Long-term (> 7 days):** STL seasonal decomposition + linear trend + historical volatility intervals

In [ ]:
def build_forecast_features(zone: str, cfg: dict,
                             start: pd.Timestamp, end: pd.Timestamp,
                             historical_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build feature matrix for forecasting.
    - Weather: use forecast API for next 7 days, then climatological averages
    - Prices: use historical lags (no future leakage)
    """
    horizon_index = pd.date_range(start, end, freq='1h', tz='UTC')
    df_fcst = pd.DataFrame(index=horizon_index)
    df_fcst['price'] = np.nan  # unknown future prices

    # Get weather forecast (7 days from now)
    days_ahead = (start - pd.Timestamp.now(tz='UTC')).days

    if days_ahead <= 6:
        print(f'  [{zone}] Using weather forecast API...')
        weather_fcst = fetch_weather_forecast(cfg['lat'], cfg['lon'])
    else:
        print(f'  [{zone}] Beyond forecast horizon — using climatological weather...')
        # Use same calendar period from previous years as weather proxy
        weather_fcst = _climatological_weather(historical_df, horizon_index)

    # Align weather to forecast index
    df_fcst = df_fcst.join(
        weather_fcst.reindex(horizon_index, method='nearest'),
        how='left'
    )

    # Concatenate with historical so we can compute lags
    combined = pd.concat([historical_df, df_fcst])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()

    # Apply feature engineering
    combined_feat = make_features(combined, cfg['country'])

    # Return only the forecast window
    return combined_feat.loc[horizon_index]


def _climatological_weather(hist: pd.DataFrame, target_idx: pd.DatetimeIndex) -> pd.DataFrame:
    """Return mean weather for the same calendar dates from historical data."""
    weather_cols = [c for c in hist.columns if c in WEATHER_VARS]
    result = pd.DataFrame(index=target_idx, columns=weather_cols, dtype=float)
    for ts in target_idx:
        # Average over same month/day/hour in historical data
        mask = (
            (hist.index.month == ts.month) &
            (hist.index.day == ts.day) &
            (hist.index.hour == ts.hour)
        )
        result.loc[ts, weather_cols] = hist.loc[mask, weather_cols].mean()
    return result


def longterm_forecast_stl(zone: str, historical_prices: pd.Series,
                           horizon_index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    Long-term fallback: STL decomposition + linear trend.
    Intervals use historical volatility (widened with sqrt(horizon)).
    """
    # Use last 2 years of data for STL
    hist = historical_prices.dropna().last('730D').resample('1h').mean().ffill()

    # STL decomposition (period = 24*7 = weekly seasonality on hourly data)
    # For speed, downsample to daily for long-term
    daily = hist.resample('D').mean().ffill()
    stl = STL(daily, period=7, robust=True)
    result = stl.fit()

    # Fit linear trend on the trend component
    trend = result.trend
    x = np.arange(len(trend))
    slope, intercept = np.polyfit(x, trend.values, 1)

    # Historical seasonal profile (hourly, by month and hour)
    seasonal_profile = hist.groupby([hist.index.month, hist.index.hour]).mean()
    overall_mean = hist.mean()

    # Build forecast
    records = []
    n_hist_days = len(daily)
    hist_vol = daily.std()

    for i, ts in enumerate(horizon_index):
        days_out = (ts - hist.index[-1]).days

        # Trend extrapolation
        trend_val = intercept + slope * (n_hist_days + days_out)

        # Seasonal adjustment
        try:
            seas_adj = seasonal_profile.get((ts.month, ts.hour), overall_mean) - overall_mean
        except:
            seas_adj = 0

        p50 = trend_val + seas_adj

        # Uncertainty widens with sqrt(horizon_days)
        uncertainty = 1.96 * hist_vol * np.sqrt(max(1, days_out) / 30)

        records.append({
            'timestamp': ts,
            'p025': p50 - uncertainty,
            'p50': p50,
            'p975': p50 + uncertainty,
        })

    return pd.DataFrame(records).set_index('timestamp')


def predict_zone(zone: str, cfg: dict,
                 start: pd.Timestamp, end: pd.Timestamp,
                 historical_df: pd.DataFrame,
                 models: dict, feature_cols: list,
                 short_term_days: int = 7) -> pd.DataFrame:
    """Regime-aware forecast for one zone."""
    now = pd.Timestamp.now(tz='UTC')
    horizon_index = pd.date_range(start, end, freq='1h', tz='UTC')

    short_cutoff = now + pd.Timedelta(days=short_term_days)
    short_idx = horizon_index[horizon_index <= short_cutoff]
    long_idx  = horizon_index[horizon_index >  short_cutoff]

    results = []

    # ── Short-term: LightGBM ───────────────────────────────────────────────────
    if len(short_idx) > 0:
        print(f'  [{zone}] Short-term: {len(short_idx)} hours via LightGBM')
        X_fcst = build_forecast_features(zone, cfg, short_idx[0], short_idx[-1], historical_df)

        # Fill any missing feature cols
        for col in feature_cols:
            if col not in X_fcst.columns:
                X_fcst[col] = 0.0
        X_fcst = X_fcst[feature_cols].fillna(method='ffill').fillna(0)

        for ts in short_idx:
            row = X_fcst.loc[[ts]] if ts in X_fcst.index else X_fcst.iloc[[-1]]
            results.append({
                'timestamp': ts,
                'p025': float(models['p025'].predict(row)[0]),
                'p50':  float(models['p50'].predict(row)[0]),
                'p975': float(models['p975'].predict(row)[0]),
                'regime': 'short_term',
            })

    # ── Long-term: STL ────────────────────────────────────────────────────────
    if len(long_idx) > 0:
        print(f'  [{zone}] Long-term: {len(long_idx)} hours via STL decomposition')
        lt = longterm_forecast_stl(zone, historical_df['price'], long_idx)
        for ts in long_idx:
            row = lt.loc[ts] if ts in lt.index else lt.iloc[-1]
            results.append({
                'timestamp': ts,
                'p025': float(row['p025']),
                'p50':  float(row['p50']),
                'p975': float(row['p975']),
                'regime': 'long_term',
            })

    df_out = pd.DataFrame(results).set_index('timestamp').sort_index()

    # Sanity check: p025 ≤ p50 ≤ p975
    df_out['p025'] = np.minimum(df_out['p025'], df_out['p50'])
    df_out['p975'] = np.maximum(df_out['p975'], df_out['p50'])

    return df_out


print('Forecasting engine defined.')

## 8. Long-term Forecast Visualization (2 years)

In [ ]:
lt_start = pd.Timestamp.now(tz='UTC').ceil('1h')
lt_end   = lt_start + pd.DateOffset(years=2)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.suptitle('2-Year Electricity Price Forecast', fontsize=14, fontweight='bold')

lt_forecasts = {}
for ax, zone in zip(axes, ZONES):
    cfg = ZONES[zone]
    print(f'Generating 2-year forecast for {zone}...')

    # For 2-year horizon, use STL directly
    lt_idx = pd.date_range(lt_start, lt_end, freq='D', tz='UTC')  # daily for speed
    lt_pred = longterm_forecast_stl(zone, feat_data[zone]['price'], lt_idx)
    lt_forecasts[zone] = lt_pred

    color = 'steelblue' if zone == 'DE-LU' else 'firebrick'
    ax.fill_between(lt_pred.index, lt_pred['p025'], lt_pred['p975'],
                    alpha=0.2, color=color, label='95% interval')
    ax.plot(lt_pred.index, lt_pred['p50'], color=color, linewidth=1.5, label='p50 forecast')

    # Historical reference (last 6 months)
    hist = feat_data[zone]['price'].last('180D').resample('D').mean()
    ax.plot(hist.index, hist.values, color='gray', linewidth=1, alpha=0.5, label='Historical')

    ax.set_title(f'{zone}')
    ax.set_ylabel('EUR/MWh')
    ax.legend(loc='upper right')
    ax.axhline(0, color='k', linestyle=':', linewidth=0.5)

plt.tight_layout()
plt.savefig('longterm_forecast.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Generate Evaluation Window Predictions

In [ ]:
print(f'Generating predictions for evaluation window:')
print(f'  {EVAL_START} → {EVAL_END}')

eval_preds = {}
for zone in ZONES:
    cfg = ZONES[zone]
    pred = predict_zone(
        zone=zone,
        cfg=cfg,
        start=EVAL_START,
        end=EVAL_END,
        historical_df=feat_data[zone],
        models=all_models[zone],
        feature_cols=all_feature_cols[zone],
        short_term_days=7,
    )
    eval_preds[zone] = pred
    print(f'  {zone}: {len(pred)} rows | regime counts: {pred["regime"].value_counts().to_dict()}')

In [ ]:
# ─── Build submission CSV ─────────────────────────────────────────────────────
TEAM_NAME = 'your_team_name'  # <-- change this!

# The evaluation window in CEST (UTC+2)
# Must have exactly 30 rows: 2026-05-08 18:00 CEST to 2026-05-09 23:00 CEST
eval_timestamps_utc = pd.date_range(EVAL_START, EVAL_END, freq='1h', tz='UTC')
assert len(eval_timestamps_utc) == 30, f'Expected 30 hours, got {len(eval_timestamps_utc)}'

rows = []
for ts_utc in eval_timestamps_utc:
    # Format as ISO 8601 with CEST offset (+02:00)
    ts_cest = ts_utc.tz_convert('Europe/Berlin')
    ts_str = ts_cest.isoformat()

    de_row = eval_preds['DE-LU'].loc[ts_utc] if ts_utc in eval_preds['DE-LU'].index else eval_preds['DE-LU'].iloc[-1]
    es_row = eval_preds['ES'].loc[ts_utc] if ts_utc in eval_preds['ES'].index else eval_preds['ES'].iloc[-1]

    rows.append({
        'timestamp':    ts_str,
        'DE-LU p025':  round(float(de_row['p025']), 2),
        'DE-LU p50':   round(float(de_row['p50']),  2),
        'DE-LU p975':  round(float(de_row['p975']), 2),
        'ES p025':     round(float(es_row['p025']), 2),
        'ES p50':      round(float(es_row['p50']),  2),
        'ES p975':     round(float(es_row['p975']), 2),
    })

submission = pd.DataFrame(rows)

# Validate
assert len(submission) == 30, f'Wrong row count: {len(submission)}'
assert list(submission.columns) == ['timestamp', 'DE-LU p025', 'DE-LU p50', 'DE-LU p975', 'ES p025', 'ES p50', 'ES p975']
assert submission.isnull().sum().sum() == 0, 'Missing values in submission!'

fname = f'{TEAM_NAME}_predictions.csv'
submission.to_csv(fname, index=False)
print(f'\nSubmission saved: {fname}')
print(f'Rows: {len(submission)}')
submission.head()

In [ ]:
# ─── Visualize evaluation window predictions ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Evaluation Window Predictions (2026-05-08 18:00 → 2026-05-09 23:00 CEST)', fontsize=12)

for ax, zone, color in zip(axes, ['DE-LU', 'ES'], ['steelblue', 'firebrick']):
    p = eval_preds[zone]
    ax.fill_between(p.index, p['p025'], p['p975'], alpha=0.25, color=color, label='p2.5–p97.5')
    ax.plot(p.index, p['p50'], color=color, linewidth=2.5, label='p50 (point forecast)')
    ax.set_title(zone)
    ax.set_ylabel('EUR/MWh')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eval_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Evaluation prediction plot saved.')

## 10. Summary & README for Data Zip

In [ ]:
readme_content = """# Data README

## Files

### DE-LU_prices.csv
- Source: ENTSO-E Transparency Platform (https://transparency.entsoe.eu)
  or energy-charts.info (https://energy-charts.info/charts/price_spot_market)
- Variables: timestamp (UTC), price (EUR/MWh)
- Frequency: Hourly
- Date range: 2022-01-01 to 2026-05-07
- Zone: DE-LU (Germany-Luxembourg, EIC: 10Y1001A1001A82H)

### ES_prices.csv
- Source: ENTSO-E Transparency Platform or energy-charts.info
- Variables: timestamp (UTC), price (EUR/MWh)
- Frequency: Hourly
- Date range: 2022-01-01 to 2026-05-07
- Zone: ES (Spain, EIC: 10YES-REE------0)

### DE-LU_weather.csv
- Source: Open-Meteo ERA5 reanalysis (https://open-meteo.com)
- Variables: temperature_2m, wind_speed_10m, wind_speed_100m,
             shortwave_radiation, direct_radiation, cloud_cover
- Coordinates: lat=51.5, lon=10.0 (centroid of DE-LU zone)
- Frequency: Hourly UTC
- Date range: 2022-01-01 to 2026-05-07

### ES_weather.csv
- Source: Open-Meteo ERA5 reanalysis (https://open-meteo.com)
- Variables: Same as DE-LU_weather.csv
- Coordinates: lat=40.4, lon=-3.7 (Madrid, centroid of ES zone)
- Frequency: Hourly UTC
- Date range: 2022-01-01 to 2026-05-07

## Notes
- All timestamps are UTC
- Price data is Day-Ahead Auction clearing prices
- Weather data is ERA5 reanalysis for historical, Open-Meteo forecast API for future
- No fuel price data included in this version (extension: EIA API for gas/coal)
"""

os.makedirs('data', exist_ok=True)
with open('data/README.txt', 'w') as f:
    f.write(readme_content)
print('data/README.txt written.')